# Signal-discovery results: browse and compare

Choose discovery runs to compare ranked signals against the complete human qrels. The notebook reports macro-averaged nDCG at several cutoffs and lets you inspect each condition's ranked signals.

Runs record their `signal_catalog`. **Path-aware** evaluates feature plus provenance path, including relationship signals. **Feature-collapsed** evaluates one direct candidate per physical table/column and uses only the matching direct-path qrels. Do not compare nDCG between these candidate universes as if they were the same task; compare methods within each view.

Human qrels are the independent headline evaluation. LLM discovery runs store a 0--100 utility score and short evidence note; this score is a ranking signal, not a qrel grade.

In [6]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import clear_output, display, Markdown
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None  # Static tables remain usable when ipywidgets is unavailable.
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None  # Tables and browsing still work without the optional plot dependency.

REPO = Path("/storage/lcostes/Similarity_benchmark")
if not (REPO / "src/evaluation/ranking_metrics.py").is_file():
    raise FileNotFoundError(f"Similarity_benchmark code was not found under {REPO}")
sys.path.insert(0, str(REPO))
from src.evaluation.ranking_metrics import ranking_metrics

# Select one suite for diagnosis, or 'all' for pooled metrics across v1, v2, and v3.
# Each suite has its own complete human qrels; runs are always paired with matching labels.
CONDITION_SET = "all"  # choose: all, v1, v2, or v3
RUN_DIRS_BY_CONDITION_SET = {
    "v1": REPO / "results/rel-amazon/discovery/condition-v1-catalog-views",
    "v2": REPO / "results/rel-amazon/discovery/condition-v2-catalog-views",
    "v3": REPO / "results/rel-amazon/discovery/condition-v3-catalog-views",
}

QREL_FILES_BY_CONDITION_SET = {
    "v1": {"Human — Lysandre": REPO / "annotations/rel-amazon/human/conditions-v1/lysandre.jsonl"},
    "v2": {"Human — Lysandre": REPO / "annotations/rel-amazon/human/conditions-v2/lysandre.jsonl"},
    "v3": {"Human — Lysandre": REPO / "annotations/rel-amazon/human/conditions-v3/lysandre.jsonl"},
}
if CONDITION_SET not in {"all", *QREL_FILES_BY_CONDITION_SET}:
    raise ValueError(f"Unknown CONDITION_SET={CONDITION_SET!r}; choose all, v1, v2, or v3")
ACTIVE_CONDITION_SETS = list(QREL_FILES_BY_CONDITION_SET) if CONDITION_SET == "all" else [CONDITION_SET]

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]

INCLUDED_METHODS = {"catalog-rank", "graph-search", "graph-search-oracle-root"}
# MiniLM is retained as an implementation option but intentionally excluded from this comparison notebook.
INCLUDED_MODELS = {"random", "strands-decider", "mistral-small-3.2", "muse-glimmer-30b", "gemma-4-31b", "gpt-oss-120b", "gpt-6-luna"}
all_result_paths = [
    (suite, result_path)
    for suite in ACTIVE_CONDITION_SETS
    for result_path in sorted(RUN_DIRS_BY_CONDITION_SET[suite].glob("*.jsonl"))
]
result_paths = []
result_suite_by_path = {}
for suite, result_path in all_result_paths:
    run_rows = read_jsonl(result_path)
    run_configs = {(str(run.get("method", "unknown")), str(run.get("model", "unknown"))) for run in run_rows}
    if run_configs and all(method in INCLUDED_METHODS and model in INCLUDED_MODELS for method, model in run_configs):
        result_paths.append(result_path)
        result_suite_by_path[str(result_path)] = suite
qrel_files = {
    name: {suite: sources[name] for suite, sources in QREL_FILES_BY_CONDITION_SET.items() if suite in ACTIVE_CONDITION_SETS and name in sources and sources[name].exists()}
    for name in set().union(*(QREL_FILES_BY_CONDITION_SET[suite] for suite in ACTIVE_CONDITION_SETS))
}
# Keep only sources available for every active suite, so a pooled metric has identical coverage.
qrel_files = {name: paths for name, paths in qrel_files.items() if set(paths) == set(ACTIVE_CONDITION_SETS)}
if not result_paths:
    searched = ', '.join(str(RUN_DIRS_BY_CONDITION_SET[suite]) for suite in ACTIVE_CONDITION_SETS)
    raise FileNotFoundError(f"No discovery JSONL files for {ACTIVE_CONDITION_SETS}. Searched: {searched}")
if not qrel_files:
    raise FileNotFoundError("No configured qrel files were found; check QREL_FILES.")
human_rows = [row for path in qrel_files.get("Human — Lysandre", {}).values() for row in read_jsonl(path)]
human_complete = sum(row.get("status") in {"judged", "complete"} and type(row.get("grade")) is int for row in human_rows)
print(f"{len(result_paths)} discovery run(s), {len(qrel_files)} qrel source(s)")
print(f"Condition suite: {CONDITION_SET} | Human qrels: {human_complete}/{len(human_rows)} complete")
print("Result folders: " + ', '.join(str(RUN_DIRS_BY_CONDITION_SET[suite]) for suite in ACTIVE_CONDITION_SETS))

96 discovery run(s), 1 qrel source(s)
Condition suite: all | Human qrels: 585/585 complete
Result folders: /storage/lcostes/Similarity_benchmark/results/rel-amazon/discovery/condition-v1-catalog-views, /storage/lcostes/Similarity_benchmark/results/rel-amazon/discovery/condition-v2-catalog-views, /storage/lcostes/Similarity_benchmark/results/rel-amazon/discovery/condition-v3-catalog-views


In [7]:
def load_qrels(path, signal_catalog='path-aware'):
    rows = read_jsonl(path)
    expected, grades = {}, {}
    for row in rows:
        condition_id, candidate_id = row.get("condition_id"), row.get("candidate_id")
        if not isinstance(condition_id, str) or not isinstance(candidate_id, str):
            continue
        candidate = row.get("candidate")
        is_direct_attribute = (
            isinstance(candidate, dict)
            and candidate.get("kind") == "attribute"
            and isinstance(candidate.get("path"), list)
            and len(candidate["path"]) == 1
        )
        if signal_catalog == "feature-collapsed" and not is_direct_attribute:
            continue
        expected.setdefault(condition_id, set()).add(candidate_id)
        grade = row.get("grade")
        if row.get("status") in {"judged", "complete"} and type(grade) is int:
            grades.setdefault(condition_id, {})[candidate_id] = grade
    return grades, expected

METHOD_LABELS = {
    'catalog-rank': 'Catalog rank',
    'graph-search': 'Graph search',
    'graph-search-oracle-root': 'Graph search (oracle root)',
}
MODEL_LABELS = {
    'random': 'Random',
    'minilm': 'MiniLM',
    'strands-decider': 'Strands Decider',
    'mistral-small-3.2': 'Mistral Small 3.2',
    'muse-glimmer-30b': 'Muse Glimmer 30B',
    'gemma-4-31b': 'Gemma 4 31B',
    'gpt-oss-120b': 'GPT-OSS 120B',
    'gpt-6-luna': 'GPT-6 Luna',
}

def run_label(method, model, model_id=None):
    method_name = METHOD_LABELS.get(str(method), str(method))
    model_name = MODEL_LABELS.get(str(model), str(model))
    return f'{method_name} — {model_name}'

def record_run_label(run):
    label = run_label(run.get('method'), run.get('model'), run.get('model_id'))
    if run.get('method') in {'graph-search', 'graph-search-oracle-root'}:
        config = run.get('graph_search_config') or {}
        budget = (config.get('max_decisions'), config.get('branch_width'), config.get('max_signals'))
        if all(value is not None for value in budget):
            label += f' — d{budget[0]}/w{budget[1]}/s{budget[2]}'
    return label

def qrel_path_for_run(qrel_name, result_path):
    suite = result_suite_by_path[str(result_path)]
    return qrel_files[qrel_name][suite]

def evaluate_run(result_path, qrel_name):
    qrel_path = qrel_path_for_run(qrel_name, result_path)
    suite = result_suite_by_path[str(result_path)]
    result_rows = read_jsonl(result_path)
    condition_rows = []
    for run in result_rows:
        condition_id = run.get("condition_id")
        signal_catalog = run.get("signal_catalog", "path-aware")
        judgments, expected = load_qrels(qrel_path, signal_catalog)
        grades = judgments.get(condition_id, {})
        if not grades:
            continue
        all_ids = expected.get(condition_id, set())
        coverage = len(grades) / len(all_ids) if all_ids else 0.0
        signals = run.get("signals") or []
        ranked = [item["id"] for item in signals if isinstance(item, dict) and isinstance(item.get("id"), str)]
        # Exclude unjudged IDs; never silently score them as grade 0.
        ranked_judged = [candidate_id for candidate_id in ranked if candidate_id in grades]
        metrics = ranking_metrics(ranked_judged, grades)
        condition_rows.append({
            "condition_set": suite,
            "condition_id": condition_id,
            "condition": run.get("condition", ""),
            "method": run.get("method", "unknown"),
            "model": run.get("model", "unknown"),
            "model_id": run.get("model_id"),
            "run": record_run_label(run),
            "signal_catalog": signal_catalog,
            "qrel_coverage": coverage,
            "ranked_count": len(ranked),
            **metrics,
        })
    if not condition_rows:
        return None, pd.DataFrame()
    per_condition = pd.DataFrame(condition_rows)
    metric_columns = [
        column for column in per_condition.columns
        if column.startswith("ndcg@")
    ]
    summary = {column: per_condition[column].mean() for column in metric_columns}
    summary.update({
        "method": str(condition_rows[0]["method"]),
        "model": str(condition_rows[0]["model"]),
        "model_id": condition_rows[0]["model_id"],
        "condition_sets": ', '.join(sorted(per_condition["condition_set"].unique())),
        "signal_catalog": str(condition_rows[0]["signal_catalog"]),
        "run": condition_rows[0]["run"],
        "conditions": len(condition_rows),
        "mean_qrel_coverage": per_condition["qrel_coverage"].mean(),
    })
    return summary, per_condition

def make_comparison(qrel_name, selected_paths):
    rows, per_condition = [], []
    for path in selected_paths:
        summary, details = evaluate_run(path, qrel_name)
        if summary is not None:
            rows.append(summary)
            per_condition.append(details)
    return (
        pd.DataFrame(rows),
        pd.concat(per_condition, ignore_index=True) if per_condition else pd.DataFrame(),
    )

## Compare runs

When `CONDITION_SET` is `all`, all result suites are selected and the comparison reports their overall mean. When it is `v1`, `v2`, or `v3`, the selectors contain only that suite's runs. nDCG rewards putting higher-grade signals near the top; the different cutoffs show how ranking quality changes as more results are considered. Results are averaged across conditions; qrel coverage is the fraction of candidate labels available in the selected judgment file. The oracle root is a target-entity proxy for this diagnostic, not a human-validated gold starting table.

Path-aware and feature-collapsed runs have separate selectors and plots; no plot mixes the two candidate catalogs. Within each catalog, the comparison plots show selected runs against the matching human labels.

In [8]:
def result_run_label(path):
    rows = read_jsonl(path)
    if not rows:
        return Path(path).name
    run = rows[0]
    catalog = run.get('signal_catalog', 'path-aware')
    return f"{record_run_label(run)} — {catalog}"

comparison_result_paths = [
    path for path in result_paths
    if CONDITION_SET == 'all' or result_suite_by_path[str(path)] == CONDITION_SET
]

run_options = [
    (f"{result_run_label(path)} [{result_suite_by_path[str(path)]}]" if CONDITION_SET == 'all' else result_run_label(path), str(path))
    for path in comparison_result_paths
]
CATALOG_ORDER = ['path-aware', 'feature-collapsed']
CATALOG_TITLES = {
    'path-aware': 'Path-aware view — feature plus provenance path',
    'feature-collapsed': 'Feature-collapsed view — direct physical attributes only',
}

def render_catalog_plot(all_source_summary, catalog_view, metric_cols):
    catalog_rows = all_source_summary[all_source_summary['signal_catalog'].eq(catalog_view)].copy()
    if catalog_rows.empty or plt is None:
        return
    source_names = list(qrel_files)
    fig, axes = plt.subplots(1, len(source_names), figsize=(12 * len(source_names), 9), sharey=True)
    if len(source_names) == 1:
        axes = [axes]
    metric_names = [column.upper() for column in metric_cols]
    for ax, source_name in zip(axes, source_names):
        source_rows = catalog_rows[catalog_rows['label source'].eq(source_name)].copy()
        source_rows = source_rows.sort_values(['ndcg@5', 'run'], ascending=[False, True]).reset_index(drop=True)
        x = list(range(len(source_rows)))
        bar_width = 0.8 / max(len(metric_cols), 1)
        for metric_index, metric in enumerate(metric_cols):
            offset = (metric_index - (len(metric_cols) - 1) / 2) * bar_width
            positions = [value + offset for value in x]
            ax.bar(positions, source_rows[metric], width=bar_width, label=metric.upper())
        ax.set_title(source_name)
        ax.set_xticks(x, source_rows['run'], rotation=35, ha='right')
        ax.set_ylim(0, 1)
        ax.set_ylabel('Macro nDCG')
        ax.grid(axis='y', alpha=0.25)
        ax.set_axisbelow(True)
        ax.legend(title='Cutoff')
    fig.suptitle(CATALOG_TITLES.get(catalog_view, catalog_view))
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    plt.show()

def render_comparison(qrel_name, selected_paths, catalog_view):
    summary, _ = make_comparison(qrel_name, selected_paths)
    if summary.empty:
        print('No comparable result rows found.')
        return
    summary = summary[summary['signal_catalog'].eq(catalog_view)].copy()
    if summary.empty:
        print(f'No comparable runs for {CATALOG_TITLES.get(catalog_view, catalog_view)}.')
        return
    all_source_rows = []
    for source_name in qrel_files:
        for run_path in selected_paths:
            source_summary, _ = evaluate_run(run_path, source_name)
            if source_summary is not None:
                all_source_rows.append({'label source': source_name, **source_summary})
    all_source_summary = pd.DataFrame(all_source_rows)
    # Pool same-method results across condition suites: each bar is one overall score.
    metric_columns = [column for column in all_source_summary.columns if column.startswith('ndcg@')]
    group_columns = ['label source', 'run', 'method', 'model', 'model_id', 'signal_catalog']
    all_source_summary = (
        all_source_summary.groupby(group_columns, dropna=False, as_index=False)
        .agg({**{column: 'mean' for column in metric_columns}, 'conditions': 'sum', 'mean_qrel_coverage': 'mean'})
    )
    metric_cols = sorted(
        [column for column in summary.columns if column.startswith('ndcg@')],
        key=lambda name: int(name.split('@')[1]),
    )
    print(Markdown(f"## {CATALOG_TITLES.get(catalog_view, catalog_view)} — {qrel_name}"))
    render_catalog_plot(all_source_summary, catalog_view, metric_cols)

def result_catalog(path):
    views = {row.get('signal_catalog', 'path-aware') for row in read_jsonl(path)}
    if len(views) > 1:
        raise ValueError(f'{path} contains more than one signal_catalog: {sorted(views)}')
    return next(iter(views), 'path-aware')

run_options_by_catalog = {
    view: [(label, path) for label, path in run_options if result_catalog(path) == view]
    for view in CATALOG_ORDER
}

# For pooled results, group the matching v1/v2/v3 files into one selectable
# experiment. This keeps each plotted bar an overall, pooled result.
def pooled_run_options(catalog_view):
    grouped = {}
    for path in result_paths:
        if result_catalog(path) != catalog_view:
            continue
        label = result_run_label(path)
        grouped.setdefault(label, []).append(path)
    return [(label, tuple(sorted(paths))) for label, paths in sorted(grouped.items())]

if widgets is None:
    print("ipywidgets is unavailable; rendering two independent catalog sections against Human qrels.")
    qrel_name = next((name for name in qrel_files if name.startswith("Human")), next(iter(qrel_files)))
    for catalog_view in CATALOG_ORDER:
        render_comparison(qrel_name, [path for _, path in run_options_by_catalog[catalog_view]], catalog_view)
else:
    def catalog_dashboard(catalog_view):
        options = pooled_run_options(catalog_view) if CONDITION_SET == 'all' else run_options_by_catalog[catalog_view]
        human_default = next((name for name in qrel_files if name.startswith("Human")), next(iter(qrel_files)))
        label_picker = widgets.Dropdown(options=list(qrel_files), value=human_default, description="Labels:")
        selected_runs = tuple(value for _, value in options)
        runs_picker = widgets.SelectMultiple(
            options=options,
            value=selected_runs,
            description="Experiments:",
            layout=widgets.Layout(width="80%", height="180px"),
        )
        output = widgets.Output()
        def refresh(change=None):
            with output:
                clear_output(wait=True)
                selected = runs_picker.value
                # A pooled widget value is a tuple of v1/v2/v3 paths; a
                # single-suite widget value is one path. Flatten both forms.
                paths = [path for value in selected for path in (value if isinstance(value, tuple) else (value,))]
                render_comparison(label_picker.value, paths, catalog_view)
        label_picker.observe(refresh, names="value")
        runs_picker.observe(refresh, names="value")
        controls = [widgets.HTML(f"<h3>{CATALOG_TITLES[catalog_view]}</h3>"), label_picker, runs_picker]
        panel = widgets.VBox([*controls, output])
        refresh()
        return panel
    display(widgets.VBox([catalog_dashboard(view) for view in CATALOG_ORDER]))

## Grades of top-ranked signals for a selected method/model run

Select a method/model run and condition to see its top five signals with Human, GPT-6 Luna, and Muse Glimmer grades. The cell also summarizes how many signals each source gave grades 3, 2, 1, and 0, and lists the selected run's rank positions for every grade-3 signal. For graph-search methods, positions are within the discovered shortlist.

In [9]:
TOP_SIGNALS_TO_SHOW = 5

def _run_label(run):
    return f"{record_run_label(run)} — {run.get('signal_catalog', 'path-aware')}"

top_run_options = []
for result_path in result_paths:
    run_rows = read_jsonl(result_path)
    if not run_rows:
        continue
    label = _run_label(run_rows[0])
    suite_label = f" [{result_suite_by_path[str(result_path)]}]" if CONDITION_SET == 'all' else ''
    top_run_options.append((label + suite_label, str(result_path)))

def render_top_signal_grades(result_path, condition_id):
    run = next((row for row in read_jsonl(result_path) if row.get("condition_id") == condition_id), None)
    if run is None:
        print("No results for this condition in the selected run.")
        return
    display(Markdown(f"### {_run_label(run)} — {condition_id}: {run.get('condition', '')}"))
    catalog_view = run.get("signal_catalog", "path-aware")
    suite = result_suite_by_path[str(result_path)]
    qrel_grades = {
        label: load_qrels(suite_paths[suite], catalog_view)[0].get(condition_id, {})
        for label, suite_paths in qrel_files.items()
    }
    display(Markdown("</br>"))
    display(Markdown("#### Grade distribution for all labeled signals in this condition"))
    grade_counts = []
    for label, grades in qrel_grades.items():
        grade_counts.append({
            "label source": label,
            "grade 3": sum(grade == 3 for grade in grades.values()),
            "grade 2": sum(grade == 2 for grade in grades.values()),
            "grade 1": sum(grade == 1 for grade in grades.values()),
            "grade 0": sum(grade == 0 for grade in grades.values()),
        })
    display(pd.DataFrame(grade_counts).set_index("label source"))
    rows = []
    ranked_signals = run.get("signals") or []
    for rank, signal in enumerate(ranked_signals[:TOP_SIGNALS_TO_SHOW], 1):
        candidate_id = signal.get("id", "")
        item = {
            "rank": rank,
            "signal ID": candidate_id,
            "signal meaning": signal.get("human_explanation", ""),
        }
        for label, grades in qrel_grades.items():
            item[f"{label} grade"] = grades.get(candidate_id, "unjudged")
        rows.append(item)
    if rows:
        display(pd.DataFrame(rows))
    else:
        print("This run returned no ranked signals for the selected condition.")
    grade_three_positions = []
    for rank, signal in enumerate(ranked_signals, 1):
        candidate_id = signal.get("id", "")
        for label, grades in qrel_grades.items():
            if grades.get(candidate_id) == 3:
                grade_three_positions.append({
                    "rank in run": rank,
                    "label source": label,
                    "signal ID": candidate_id,
                    "signal meaning": signal.get("human_explanation", ""),
                })
    display(Markdown("#### Positions of grade-3 signals in this run's ranking"))
    if grade_three_positions:
        display(pd.DataFrame(grade_three_positions).sort_values(["rank in run", "label source"]).reset_index(drop=True))
    else:
        print("No returned signal has grade 3 under these label sources for this condition.")

if widgets is None:
    print("ipywidgets is unavailable; showing the first run and condition.")
    default_path = top_run_options[0][1]
    default_condition = read_jsonl(default_path)[0]["condition_id"]
    render_top_signal_grades(default_path, default_condition)
else:
    top_method_picker = widgets.Dropdown(options=top_run_options, description="Run:", layout=widgets.Layout(width="75%"))
    top_condition_picker = widgets.Dropdown(description="Condition:", layout=widgets.Layout(width="75%"))
    top_grade_output = widgets.Output()
    def refresh_top_signal_grades(change=None):
        if top_method_picker.value is None or top_condition_picker.value is None:
            return
        with top_grade_output:
            clear_output(wait=True)
            render_top_signal_grades(top_method_picker.value, top_condition_picker.value)
    def update_top_condition_options(change=None):
        run_rows = read_jsonl(top_method_picker.value)
        options = [(f"{row.get('condition_id')} — {row.get('condition', '')}", row.get("condition_id")) for row in run_rows]
        top_condition_picker.options = options
        if options:
            top_condition_picker.value = options[0][1]
        refresh_top_signal_grades()
    top_method_picker.observe(update_top_condition_options, names="value")
    top_condition_picker.observe(refresh_top_signal_grades, names="value")
    display(widgets.VBox([top_method_picker, top_condition_picker, top_grade_output]))
    update_top_condition_options()

## Browse one ranked list

Select a run and condition to inspect candidate IDs, plain-language meaning, technical signal/path, score, and qrel grade. The graph-search method returns a shortlist; it does not rank undiscovered candidates.

In [10]:
def render_ranked_list(result_path, condition_id, qrel_name):
    rows = read_jsonl(result_path)
    run = next((row for row in rows if row.get("condition_id") == condition_id), None)
    if run is None:
        print("Choose a result and condition.")
        return
    suite = result_suite_by_path[str(result_path)]
    grades, _ = load_qrels(qrel_files[qrel_name][suite], run.get("signal_catalog", "path-aware"))
    condition_grades = grades.get(run["condition_id"], {})
    display(Markdown(f"### {run['condition_id']} — {run.get('condition', '')}"))
    items = []
    for rank, signal in enumerate(run.get("signals", []), 1):
        candidate_id = signal.get("id", "")
        items.append({"rank": rank, "qrel grade": condition_grades.get(candidate_id, "unjudged"), "kind": signal.get("kind"), "source": signal.get("source_table"), "candidate": candidate_id, "signal meaning": signal.get("human_explanation", ""), "technical path": signal.get("description", ""), "score": signal.get("score"), "utility score (0-100)": signal.get("utility_score"), "LLM evidence": signal.get("evidence", "")})
    display(pd.DataFrame(items))
    if run.get("graph_search"):
        display(Markdown("#### Graph-search trace"))
        display(pd.json_normalize(run["graph_search"], sep="."))

if widgets is None:
    print("ipywidgets is unavailable; showing the first run and first condition. Edit these variables and rerun the cell to browse.")
    browse_result_path = str(result_paths[0])
    browse_condition_id = read_jsonl(browse_result_path)[0]["condition_id"]
    browse_qrel_name = next((name for name in qrel_files if name.startswith("Human")), next(iter(qrel_files)))
    render_ranked_list(browse_result_path, browse_condition_id, browse_qrel_name)
else:
    browse_run = widgets.Dropdown(options=run_options, description="Run:", layout=widgets.Layout(width="75%"))
    browse_condition = widgets.Dropdown(description="Condition:", layout=widgets.Layout(width="75%"))
    browse_qrel_picker = widgets.Dropdown(options=list(qrel_files), value=next(iter(qrel_files)), description="Labels:")
    browse_output = widgets.Output()
    def update_condition_options(change=None):
        rows = read_jsonl(browse_run.value)
        options = [(f"{row.get('condition_id')} — {row.get('condition', '')}", row.get("condition_id")) for row in rows]
        browse_condition.options = options
        if options:
            browse_condition.value = options[0][1]
    def show_ranked_list(change=None):
        with browse_output:
            clear_output(wait=True)
            render_ranked_list(browse_run.value, browse_condition.value, browse_qrel_picker.value)
    browse_run.observe(update_condition_options, names="value")
    browse_condition.observe(show_ranked_list, names="value")
    browse_qrel_picker.observe(show_ranked_list, names="value")
    display(widgets.VBox([browse_run, browse_condition, browse_qrel_picker, browse_output]))
    update_condition_options()
    show_ranked_list()